<a href="https://colab.research.google.com/github/DymaStar/DTA_2026/blob/main/ML/DS120626__under_fitting%26cross_validation_DTA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Overfitting & underfitting. Cross-validation

## Ідея уроку

У цьому notebook демонструється:
- underfitting (недонавчання),
- overfitting (перенавчання),
- та cross-validation.

Мета:
- показати, як складність моделі впливає на accuracy;
- навчитися знаходити баланс між простою та занадто складною моделлю.

Underfitting виникає, коли модель занадто проста і не може вивчити закономірності даних.

Overfitting виникає, коли модель занадто сильно "запам’ятовує" training data та погано працює на нових даних.

Cross-validation допомагає чесніше оцінювати якість моделі.

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Щоб результати були однаковими щоразу (відтворюваність)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Бібліотеки готові ✅")

Бібліотеки готові ✅


In [10]:
m = 1200
tenure   = np.random.randint(1, 72, m)           # місяців з нами
monthly  = np.random.normal(70, 25, m).clip(15, 150)  # щомісячна оплата, $
support  = np.random.poisson(1.5, m)             # звернень у підтримку за рік
age      = np.random.randint(18, 75, m)          # вік клієнта

# Прихована логіка ризику відтоку (модель її не знає):
risk = (
    -0.05 * tenure        # довше з нами → менший ризик
    + 0.02 * monthly      # дорожчий тариф → трохи більший ризик
    + 0.45 * support      # багато звернень у підтримку → більший ризик
    - 0.01 * age          # старші клієнти трохи лояльніші
    + np.random.normal(0, 0.7, m)
)
prob = 1 / (1 + np.exp(-(risk - 0.5)))   # перетворюємо ризик на ймовірність 0..1
churn = (np.random.rand(m) < prob).astype(int)

df = pd.DataFrame({
    "tenure": tenure, "monthly": monthly.round(1),
    "support": support, "age": age, "churn": churn,
})

print("Частка клієнтів, що пішли:", f"{df['churn'].mean():.1%}")
df.head()

Частка клієнтів, що пішли: 39.3%


,tenure,monthly,support,age,churn
0,52,21.7,1,21,0
1,15,39.8,1,20,0
2,61,43.2,4,73,0
3,21,87.1,0,46,1
4,24,65.9,1,69,1


In [11]:
from sklearn.model_selection import train_test_split

X = df[['tenure', 'monthly', 'support', 'age']]  # features - ознаки

y = df['churn']  # target - ціль

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Навчальна вибірка:", X_train.shape[0], "клієнтів")
print("Тестова вибірка:", X_test.shape[0], "клієнтів")

Навчальна вибірка: 960 клієнтів
Тестова вибірка: 240 клієнтів


## Перевірка overfitting на глибокому дереві

На цьому етапі ми створюємо Decision Tree без обмеження глибини.

Таке дерево може будувати дуже багато правил і майже повністю запам’ятати training data.

Якщо train accuracy дуже висока, а test accuracy значно нижча — це ознака overfitting.

In [12]:
# Імпорт моделі дерева рішень
from sklearn.tree import DecisionTreeClassifier

# Імпорт метрики accuracy
from sklearn.metrics import accuracy_score

# Створюємо "глибоке" дерево рішень
# max_depth не вказуємо, тому дерево може рости максимально глибоко
# Це може призвести до перенавчання
deep_tree = DecisionTreeClassifier(
    random_state=RANDOM_STATE
)

# Навчаємо модель на train data
deep_tree.fit(X_train, y_train)

# Accuracy на train data
# Тут модель перевіряється на тих самих даних, на яких навчалась
acc_train = accuracy_score(
    y_train,
    deep_tree.predict(X_train)
)

# Accuracy на test data
# Тут модель перевіряється на нових даних, яких вона не бачила під час навчання
acc_test = accuracy_score(
    y_test,
    deep_tree.predict(X_test)
)

# Виведення результатів
print("Глибоке дерево без обмежень (overfitting)\n")

print(
    f"Train accuracy = {acc_train:.2%} "
    f"← модель майже ідеально запам’ятала train data"
)

print(
    f"Test accuracy  = {acc_test:.2%} "
    f"← на нових даних результат значно гірший"
)
# Різниця між train та test accuracy
# Великий розрив часто означає overfitting
print(
    f"Розрив = {(acc_train - acc_test):.2%} "
    f"← ознака перенавчання\n"
)

Глибоке дерево без обмежень (overfitting)

Train accuracy = 100.00% ← модель майже ідеально запам’ятала train data
Test accuracy  = 62.92% ← на нових даних результат значно гірший
Розрив = 37.08% ← ознака перенавчання



## Висновок

Train accuracy = 100%, тобто модель ідеально запам’ятала навчальні дані.

Test accuracy = 62.92%, тобто на нових даних модель працює значно гірше.

Це класичний приклад overfitting: модель занадто складна, добре запам’ятала train data, але не дуже добре узагальнює закономірності для нових клієнтів.
(acc_train - acc_test)

рахує:

наскільки train accuracy вища за test accuracy.

100%−62.92%≈37.08%

Це великий розрив → модель перенавчилась.

## Порівняння overfitting та нормального дерева

У цьому блоці порівнюються:
- дуже глибоке дерево без обмежень;
- простіше дерево з `max_depth=3`.

Мета:
- побачити різницю між overfitting та більш стабільною моделлю.

Якщо:
- train accuracy дуже висока,
- а test accuracy значно нижча,

це означає overfitting.

Якщо train і test accuracy близькі:
- модель краще узагальнює дані;
- працює стабільніше на нових прикладах.

In [13]:
# =========================
# ГЛИБОКЕ ДЕРЕВО (OVERFITTING)
# =========================

# Створюємо дерево без обмеження глибини
# Модель може будувати дуже багато правил
deep_tree = DecisionTreeClassifier(
    random_state=RANDOM_STATE
)

# Навчання моделі
deep_tree.fit(X_train, y_train)

# Accuracy на train data
acc_train = accuracy_score(
    y_train,
    deep_tree.predict(X_train)
)

# Accuracy на test data
acc_test = accuracy_score(
    y_test,
    deep_tree.predict(X_test)
)

print("ГЛИБОКЕ дерево без обмежень (перенавчання):\n")

print(
    f"Train accuracy = {acc_train:.2%} "
    f"← модель майже ідеально запам’ятала train data"
)

print(
    f"Test accuracy = {acc_test:.2%} "
    f"← на нових даних результат гірший"
)

print(
    f"Розрив = {(acc_train - acc_test):.2%} "
    f"← ознака overfitting\n"
)


# =========================
# ПРОСТІШЕ ДЕРЕВО
# =========================

# Обмежуємо глибину дерева
# Менш складна модель = менший ризик перенавчання
shallow = DecisionTreeClassifier(
    max_depth=3,
    random_state=RANDOM_STATE
)

# Навчання моделі
shallow.fit(X_train, y_train)

# Accuracy на train data
acc_train = accuracy_score(
    y_train,
    shallow.predict(X_train)
)

# Accuracy на test data
acc_test = accuracy_score(
    y_test,
    shallow.predict(X_test)
)

print("ПРОСТЕ дерево (max_depth=3):\n")

print(
    f"Train accuracy = {acc_train:.2%}"
)

print(
    f"Test accuracy = {acc_test:.2%}"
)

print(
    f"Розрив = {(acc_train - acc_test):.2%} - модель надійніші(чесніша)"
)

ГЛИБОКЕ дерево без обмежень (перенавчання):

Train accuracy = 100.00% ← модель майже ідеально запам’ятала train data
Test accuracy = 62.92% ← на нових даних результат гірший
Розрив = 37.08% ← ознака overfitting

ПРОСТЕ дерево (max_depth=3):

Train accuracy = 72.29%
Test accuracy = 59.17%
Розрив = 13.12% - модель надійніші(чесніша)


## Правило для визначення overfitting

Якщо train accuracy набагато вища за test accuracy — це ознака перенавчання.

Модель занадто добре запам’ятала навчальні дані, але гірше працює на нових даних.

### Як зменшити overfitting

Щоб зменшити перенавчання, можна:
- обмежити глибину дерева (`max_depth`);
- зменшити складність моделі;
- прибрати зайві або шумні ознаки;
- збільшити кількість даних;
- використати cross-validation для стабільнішої оцінки.

## Висновок

Глибоке дерево:
- майже ідеально працює на train data;
- але значно гірше на test data.

Це приклад overfitting:
модель запам’ятала навчальні дані замість узагальнення закономірностей.

Простіше дерево:
- має трохи нижчий train accuracy;
- але стабільніше працює на нових даних.

Невеликий розрив між train та test accuracy — ознака більш якісної моделі.

## Cross-validation

Cross-validation — це більш надійний спосіб оцінки моделі.

Замість одного train/test split:
- дані діляться на кілька частин (folds);
- модель багато разів навчається та тестується;
- після цього рахується середній accuracy.

Наприклад при `cv=5`:
- дані діляться на 5 частин;
- 4 частини використовуються для навчання;
- 1 частина — для тесту;
- процес повторюється 5 разів.

Кожна частина один раз стає test data.

Це допомагає:
- отримати стабільнішу оцінку;
- зменшити вплив випадкового split;
- краще оцінити якість моделі.

In [14]:
# Імпорт Random Forest
from sklearn.ensemble import RandomForestClassifier

# Імпорт cross-validation
from sklearn.model_selection import cross_val_score

# Cross-validation для Random Forest
scores = cross_val_score(

    # Модель Random Forest
    RandomForestClassifier(

        # Кількість дерев
        n_estimators=200,

        # Обмеження глибини дерева
        # Допомагає зменшити overfitting
        max_depth=6,

        # Фіксація випадковості
        random_state=RANDOM_STATE
    ),

    # Features
    X,

    # Target
    y,

    # Кількість folds
    cv=5,

    # Метрика оцінки
    scoring="accuracy"
)

# Accuracy для кожного fold
print("Accuracy на кожному з 5 розбиттів:\n")
print(scores)

# Середній accuracy
print(
    f"\nСередній accuracy = {scores.mean():.2%}"
)

# Стабільність результатів
print(
    f"Стандартне відхилення = {scores.std():.2%}"
)

Accuracy на кожному з 5 розбиттів:

[0.6875     0.73333333 0.65       0.675      0.6375    ]

Середній accuracy = 67.67%
Стандартне відхилення = 3.34%


In [15]:
# Accuracy для кожного fold
# np.round(scores, 3) округлює значення до 3 знаків
print(
    f"Accuracy на кожному з 5 розбиттів:\n{np.round(scores, 3)}"
)

# scores.mean() -> середній accuracy
# scores.std() -> стандартне відхилення (стабільність моделі)
print(
    f"Середнє accuracy = {scores.mean():.2%} "
    f"+- {scores.std():.2%}"
)

Accuracy на кожному з 5 розбиттів:
[0.688 0.733 0.65  0.675 0.638]
Середнє accuracy = 67.67% +- 3.34%


## Пояснення результатів

Модель Random Forest показала accuracy:
- від 63.8% до 73.3% на різних folds.

### Середнє accuracy

`67.67%` означає, що модель у середньому правильно класифікує приблизно 68% клієнтів.

### Що означає "+- 3.34%"

`3.34%` — це стандартне відхилення (`std`).

Воно показує:
- наскільки результати змінюються між різними folds;
- наскільки стабільно працює модель.

Невелике std означає:
- модель стабільна;
- accuracy не сильно залежить від випадкового train/test split.

## Висновок

Cross-validation дозволяє отримати більш чесну та стабільну оцінку моделі.

Замість одного випадкового split:
- модель тестується кілька разів;
- на різних частинах даних;
- а результати усереднюються.

Середній accuracy (`mean`) показує загальну якість моделі.

Стандартне відхилення (`std`) показує:
- наскільки стабільні результати між різними folds;
- чи сильно змінюється accuracy на різних частинах даних.